# 11 — Final Evaluation

## Objective
**Locked test-set evaluation.**  
Train the final IsolationForest on train+val, score the held-out test set, and report ranking metrics.  The test partition has remained untouched until this notebook.

Also freeze the model + feature list for the Streamlit deployment.


In [ ]:

from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import joblib

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data_utils import load_processed
from anomaly import fit_isolation_forest
from evaluation import evaluate_anomaly_scores, summary_table, enrichment_factor

X_train = load_processed("X_train")
X_val   = load_processed("X_val")
X_test  = load_processed("X_test")

y_test = X_test["class"].values
X_train_full = pd.concat([X_train.drop(columns=["class"]), X_val.drop(columns=["class"])])
X_test_f = X_test.drop(columns=["class"])

print("Fitting final IsolationForest on train+val …")
final_model = fit_isolation_forest(X_train_full, contamination=0.09, n_estimators=300, random_state=42)
scores = final_model.predict_anomaly_score(X_test_f)
metrics = evaluate_anomaly_scores(y_test, scores)
print(summary_table({"Final IsolationForest (test)": metrics}))
print("Enrichment factor @ #positives:", enrichment_factor(y_test, scores, int(y_test.sum())))

# Freeze artefacts
models_dir = Path(ROOT) / "models"
models_dir.mkdir(exist_ok=True)
joblib.dump(final_model, models_dir / "final_model.joblib")
config = {"features": list(X_train_full.columns), "model": "IsolationForest", "contamination": 0.09}
(models_dir / "inference_config.json").write_text(json.dumps(config, indent=2))
print("\nFrozen model + config written to models/")
